# Kaggle 03 - Evaluation / Ablation

Attach Kaggle Dataset `oopclone989876/ks-project2-data` and run after Qdrant Cloud is populated.


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/kaggle/working/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[qdrant,agent,eval]"
!pip install requests gdown


In [ ]:
# Load Kaggle Secrets. Add these in Notebook > Add-ons > Secrets.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        value = None
        print(f"Secret {name} unavailable: {exc}")
    if value:
        os.environ[name] = value

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("QDRANT_TEXT_COLLECTION", "text_chunks_prod")
os.environ.setdefault("QDRANT_IMAGE_COLLECTION", "image_patches_prod")
os.environ.setdefault("QDRANT_INDEX_STATE", "/kaggle/working/outputs/index_state/kaggle_index_state.json")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("Text collection:", os.environ.get("QDRANT_TEXT_COLLECTION"))
print("Image collection:", os.environ.get("QDRANT_IMAGE_COLLECTION"))


In [ ]:
# Copy data from attached Kaggle Dataset.
# In Kaggle, click Add Input and attach: oopclone989876/ks-project2-data
from pathlib import Path
import os
import shutil
import zipfile

DATASET_ROOT = Path("/kaggle/input/ks-project2-data")
PROJECT_DATA = Path("/kaggle/working/project-ks2/data")
PROJECT_DATA.mkdir(parents=True, exist_ok=True)

if not DATASET_ROOT.exists():
    raise RuntimeError(
        "Kaggle dataset is not attached. Use Add Input -> oopclone989876/ks-project2-data."
    )

print("Dataset root:", DATASET_ROOT)
print("Dataset preview:")
!find /kaggle/input/ks-project2-data -maxdepth 3 -type f | head -50

# If the dataset contains a single zip, unzip it first.
zip_files = list(DATASET_ROOT.glob("*.zip"))
if zip_files and not any((DATASET_ROOT / name).exists() for name in ["bioasq", "medqa", "eval_cases.json", "data"]):
    unzip_root = Path("/kaggle/working/dataset_unzipped")
    if unzip_root.exists():
        shutil.rmtree(unzip_root)
    unzip_root.mkdir(parents=True, exist_ok=True)
    print("Unzipping:", zip_files[0])
    with zipfile.ZipFile(zip_files[0]) as zf:
        zf.extractall(unzip_root)
    search_root = unzip_root
else:
    search_root = DATASET_ROOT

candidates = [
    search_root,
    search_root / "data",
    search_root / "KS_Project_2" / "data",
    search_root / "project-ks2-data",
    search_root / "project-ks2-data" / "data",
]

selected = None
for candidate in candidates:
    if candidate.exists() and (
        (candidate / "eval_cases.json").exists()
        or (candidate / "bioasq").exists()
        or (candidate / "medqa").exists()
        or (candidate / "roco").exists()
        or (candidate / "vqa_rad").exists()
    ):
        selected = candidate
        break

if selected is None:
    print("Could not auto-detect data folder. Tree preview:")
    !find /kaggle/input/ks-project2-data -maxdepth 5 | head -100
    raise RuntimeError("Data folder not detected in Kaggle Dataset.")

print("Selected data source:", selected)
for item in selected.iterdir():
    target = PROJECT_DATA / item.name
    if target.exists():
        if target.is_dir():
            shutil.rmtree(target)
        else:
            target.unlink()
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)

print("Copied data to:", PROJECT_DATA)
!find /kaggle/working/project-ks2/data -maxdepth 3 -type f | head -50


In [ ]:
# Audit data before indexing/evaluation. If this reports zero records, fix data setup first.
!python -m medical_rag audit-data --data-dir data || true
!python -m medical_rag project-status --data-dir data || true


In [ ]:
# Provider diagnostics
!python -m medical_rag test-openrouter
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Baseline advanced metrics
!python scripts/colab_workflow.py eval-baseline   --eval-file data/eval_cases.json   --output-file outputs/benchmark/baseline_advanced.json


In [ ]:
# Agent metrics with Qdrant Cloud + OpenRouter
!python scripts/colab_workflow.py eval-agent   --eval-file data/eval_cases.json   --output-file outputs/benchmark/agent_openrouter.json   --use-qdrant


In [ ]:
# Ablation report
!python scripts/colab_workflow.py ablation   --eval-file data/eval_cases.json   --output-dir outputs/ablation


In [ ]:
# Package state/output files for download from Kaggle Output panel.
!mkdir -p /kaggle/working/artifacts
!cp -r /kaggle/working/project-ks2/outputs /kaggle/working/artifacts/outputs || true
!find /kaggle/working/artifacts -maxdepth 4 -type f | sort | head -100
